# 01 — LangChain Foundations
### Messages, Chat Models, and Prompt Templates

This is notebook 1 of a 6-part series that teaches LangChain by walking through
the **SupportPilot** capstone project (ShopStream India customer support
automation) piece by piece. Each notebook lives in the same folder as the
project's `.py` modules, so you can `import` real project code directly.

**Series roadmap:**
1. **Foundations** — messages, chat models, prompt templates *(this notebook)*
2. Output parsers & structured output (Pydantic)
3. LCEL — chaining runnables together
4. Tools & function calling
5. Embeddings, vector stores & retrieval (RAG)
6. Putting it together — the full SupportPilot pipeline

**Setup:** run this from inside the `supportpilot_langchain/` project folder
(or copy this notebook there) so that `import database`, `import models`, etc.
resolve correctly in later notebooks.

```bash
pip install -r requirements.txt
```

No API key is required for most of this series — we use LangChain's built-in
**fake chat models** to demonstrate concepts offline, and clearly mark the
cells where a real `ANTHROPIC_API_KEY` would change the output.

## 1.1 Why LangChain?

Before LangChain, using an LLM meant writing your own glue code for every
provider's API, every prompt template, every parsing step. LangChain gives you:

- **A common interface** across model providers (`ChatAnthropic`, `ChatOpenAI`,
  etc. all implement the same `.invoke()` / `.stream()` / `.batch()` methods)
- **Prompt templates** — parameterized, reusable prompts instead of f-strings
- **Output parsers** — turning raw text into structured Python objects
- **Runnables (LCEL)** — a composition language for chaining steps together
  with `|`
- **Tools** — a standard way to expose Python functions to a model
- **Retrievers / vector stores** — standard interfaces for RAG

You already saw all of these in the SupportPilot project's `chains.py`,
`tools.py`, and `retrieval.py`. This series builds each concept up from
scratch, then shows you exactly where it's used in that code.

## 1.2 Messages

LangChain represents a conversation as a list of typed messages, not a single
string. This matters because chat models are trained on a specific
role-structured format (system / human / assistant), and getting that
structure right is what "prompt engineering" actually means in practice.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You classify ShopStream India customer support tickets."),
    HumanMessage(content="Where is my order? It's been 6 days and tracking hasn't updated."),
]

for m in messages:
    print(f"{type(m).__name__}: {m.content}")


## 1.3 Chat models — the common interface

Every chat model in LangChain — real or fake — implements the same
`.invoke(messages)` method and returns an `AIMessage`. This is what lets you
swap `ChatAnthropic` for a fake model in tests without touching any other
code, exactly like SupportPilot's `mock` vs `live` mode in `chains.py`.

LangChain ships fake chat models specifically for this kind of offline
learning/testing — no API key needed for the cells below.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

# A fake model that just returns pre-programmed responses in order,
# regardless of what you send it. Useful for understanding the .invoke()
# contract without spending real API calls.
fake_model = FakeListChatModel(responses=[
    "This looks like an order status inquiry.",
    "This looks like a refund request.",
])

response = fake_model.invoke(messages)
print(type(response))
print(response.content)


In [ ]:
# Call it again -- it returns the next response in the list
response2 = fake_model.invoke([HumanMessage(content="I want my money back")])
print(response2.content)


### What changes with a real model

Swap `FakeListChatModel` for `ChatAnthropic` and the `.invoke()` call is
*identical* — that's the point of the common interface. This is commented out
so the notebook still runs without a key:

```python
from langchain_anthropic import ChatAnthropic
import os

real_model = ChatAnthropic(model="claude-sonnet-5", max_tokens=500)
# requires: os.environ["ANTHROPIC_API_KEY"] = "sk-..."
response = real_model.invoke(messages)
print(response.content)
```

This is exactly what `chains.py`'s `_get_chat_model()` function does in
`live` mode.

## 1.4 Prompt templates

Hardcoding message text like we did above doesn't scale — you need to fill in
different ticket text, customer names, etc. every time. `ChatPromptTemplate`
lets you define a template once and `.invoke()` it with different variables.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "You classify ShopStream India customer support tickets. "
               "If the ticket is ambiguous, say so rather than guessing."),
    ("human", "ticket_id: {ticket_id}\nticket_text: {ticket_text}"),
])

# .invoke() on a prompt template returns a formatted list of messages,
# ready to hand to a chat model.
formatted = classify_prompt.invoke({
    "ticket_id": "T001",
    "ticket_text": "Where is my order? It's been 6 days.",
})
for m in formatted.to_messages():
    print(f"{type(m).__name__}: {m.content}")


Compare this to the real classification prompt in `chains.py`:

```python
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You classify ShopStream India customer support tickets accurately and conservatively. "
     "If the ticket is ambiguous, reflect that with a lower confidence score rather than guessing."),
    ("human", "ticket_id: {ticket_id}\nticket_text: {ticket_text}"),
])
```

Same pattern — a system message setting the task, a human message with
`{placeholders}` filled at call time. The only difference is what comes
*after* the prompt in the chain, which is what notebook 3 (LCEL) covers.

## 1.5 Piping a prompt into a model — your first chain

The `|` operator is LangChain's composition syntax (LCEL — LangChain
Expression Language). `prompt | model` builds a `RunnableSequence`: calling
`.invoke()` on it formats the prompt *and* calls the model in one step.

In [ ]:
chain = classify_prompt | fake_model

result = chain.invoke({"ticket_id": "T002", "ticket_text": "I'd like a refund please."})
print(result.content)


That's the whole idea behind `chains.py`'s `build_classification_chain()`,
`build_drafting_chain()`, and `build_validation_chain()` — each one is a
`prompt | model | parser` pipe, just with a real model and a parser that
turns the output into a structured object instead of leaving it as raw text.
That's exactly what notebook 2 covers next.

## Exercise

1. Write a `ChatPromptTemplate` for the **drafting** step: system message
   telling the model to write an empathetic ShopStream support response,
   human message template taking `classification` and `kb_chunks` as
   variables.
2. Pipe it into a `FakeListChatModel` with a couple of canned responses and
   invoke it with sample data.
3. Compare your prompt to the real one in `chains.py`'s
   `build_drafting_chain()` — what's different, and why might that matter for
   response quality?